# 04 · Đánh giá tổng hợp + đẩy model lên Hugging Face Hub

Input: code + output của 01 (splits, baseline, nrl), 02 (export ONNX), 03 (runs/s2s).
Trên Colab các output này đã nằm trong Drive sau khi chạy 01-03.
Bật GPU để seq2seq decode nhanh; tagger ONNX đo trên CPU 1 luồng cho giống máy yếu.
Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → Change runtime type → T4 GPU.
Secrets cần có: `HF_TOKEN` (write; Kaggle: Add-ons → Secrets, Colab: biểu tượng chìa khoá bên trái).
Đổi `HF_REPO` thành tên của bạn trên HF.

In [ ]:
import glob, json, os, sys

IN_KAGGLE = os.path.exists("/kaggle/input")
IN_COLAB = "google.colab" in sys.modules
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/vi_diacritics/__init__.py", recursive=True)[0].rsplit("/vi_diacritics", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vi-diacritics-transformer" if IN_COLAB else os.path.abspath("vi-diacritics-transformer")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vi-diacritics-transformer {REPO}
    CODE = f"{REPO}/src"
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ai-portfolio/vi-diacritics-transformer" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE


def secret(name):
    # Kaggle Secrets / Colab Secrets / biến môi trường, thiếu thì trả None
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)


def find(pat, hint):
    # output của notebook trước: Kaggle thì add làm input (/kaggle/input), Colab thì nằm sẵn trong WORK trên Drive
    for root in ("/kaggle/input", WORK, REPO):
        hits = glob.glob(f"{root}/{pat}") or glob.glob(f"{root}/**/{pat}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"không tìm thấy {pat}: {hint}")


SPLITS = os.path.dirname(find("splits/train.txt", "chạy notebook 01 trước"))
EVAL = os.path.dirname(find("eval/nrl_all.txt", "chạy notebook 01 trước"))
EXPORT = os.path.dirname(find("export/tagger.onnx", "chạy notebook 02 trước"))
S2S = os.path.dirname(find("s2s/best.pt", "chạy notebook 03 trước"))
BASELINE = find("baseline.json", "chạy notebook 01 trước")
REAL = find("real_typing.txt", "thiếu data/real_typing.txt của repo")
HF_REPO = "DuongCodeAI/vi-diacritics-tagger"
!pip install -q onnxruntime tokenizers

In [ ]:
tests = [f"wiki={SPLITS}/test.txt", f"real={REAL}"] + [
    f"{os.path.basename(p)[:-4]}={p}" for p in sorted(glob.glob(f"{EVAL}/nrl_*.txt"))]
args = " ".join(f"--test {t}" for t in tests)
!python -m vi_diacritics.evaluate --export {EXPORT} --baseline {BASELINE} --s2s-run {S2S} {args} --limit 3000 --out results

In [ ]:
print(open("results/eval.md", encoding="utf-8").read())

## Phân tích lỗi: âm tiết nào khó nhất, nhầm thành gì

In [ ]:
import pandas as pd

err = json.load(open("results/errors_wiki_tagger_int8.json", encoding="utf-8"))
display(pd.DataFrame(err["hard_syllables"], columns=["âm tiết", "lỗi", "tổng", "tỉ lệ lỗi"]).head(20))
display(pd.DataFrame(err["confusions"], columns=["đúng", "đoán", "số lần"]).head(20))

In [ ]:
err = json.load(open("results/errors_real_tagger_int8.json", encoding="utf-8"))
for e in err["examples"][:20]:
    print("IN  :", e["input"])
    print("PRED:", e["pred"])
    print("GOLD:", e["gold"])
    print()

## Đẩy ONNX lên HF Hub (để repo khác + web demo tải về)

In [ ]:
from huggingface_hub import HfApi

token = secret("HF_TOKEN")
api = HfApi(token=token)
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(folder_path=EXPORT, repo_id=HF_REPO)
api.upload_folder(folder_path="results", repo_id=HF_REPO, path_in_repo="results")
print(f"https://huggingface.co/{HF_REPO}")